In [1]:
import numpy as np
import pandas as pd
from sys import stderr
from numpy import zeros, arange, isscalar,diag, dot,eye, ix_, ones, r_, pi, flatnonzero as find
from scipy.sparse import csr_matrix
from numpy.linalg import solve
import torch

In [ ]:
# === Initialization ===
from torch import device


case_name = 'pglib_opf_case3_lmbd'
case_path = f'../excel_outputs/{case_name}.xlsx'
case = pd.read_excel(case_path, sheet_name=['baseMVA','bus','gen','gencost','branch'])

bus_to_idx = {bus: i+1 for i, bus in enumerate(case['bus'].bus_i.values)}
bus_idx = [bus_to_idx[bus] for bus in case['bus'].bus_i.values]
case['bus'].bus_i = case['bus'].bus_i.replace(bus_to_idx) # rename the bus for making PTDF
case['gen'].bus_i = case['gen'].bus_i.replace(bus_to_idx)
case['gencost'].bus_i = case['gencost'].bus_i.replace(bus_to_idx)
case['branch'].bus_i = case['branch'].bus_i.replace(bus_to_idx)
case['branch'].bus_j = case['branch'].bus_j.replace(bus_to_idx)
nbus = case['bus'].shape[0]
ngen = case['gen'].shape[0]
nbranch = case['branch'].shape[0]

# per unit p.u. conversion
baseMVA = case['baseMVA'].values[0][0]
# Pmax = case['gen'].Pmax.values / baseMVA
# Pmin = case['gen'].Pmin.values / baseMVA
# Qmax = case['gen'].Qmax.values / baseMVA
# Qmin = case['gen'].Qmin.values / baseMVA
# I_max = case['branch'].rateA.values / baseMVA
c2 = case['gencost'].c2.values * baseMVA**2
c1 = case['gencost'].c1.values * baseMVA
c0 = case['gencost'].c0.values

# calculate susceptance, conductance, admittance-square y_sq
# $Z = r + ix$ $Y = g + ib$ $Y = \frac{1}{Z} = \frac{r}{r^2 + x^2} - i\frac{x}{r^2 + x^2}$
# 1. Physics: Admittance Y = g + i*b
r = case['branch']['r'].values
x = case['branch']['x'].values
Z_sq = r**2 + x**2
g = r / Z_sq
b = -x / Z_sq

# 2. Extract Line Charging, Taps, and Phase Shifts
bc = case['branch']['b'].values # MATPOWER branch 'b' is total line charging susceptance
tau = np.where(case['branch']['ratio'].values == 0, 1.0, case['branch']['ratio'].values)
theta_shift = np.radians(case['branch']['angle'].values)

# 3. Data Extraction
Gs = case['bus']['Gs'].values / baseMVA
Bs = case['bus']['Bs'].values / baseMVA
Pd = case['bus'].Pd.values / baseMVA
Qd = case['bus'].Qd.values / baseMVA

# State vector dimension D = 2 * |B|
D = 2 * nbus

r = case['branch']['r'].values
x = case['branch']['x'].values
y_sq = 1 / (r**2 + x**2)

M_I = []
for l in range(nbranch):
    # Extract 0-based indices directly
    i = int(case['branch']['bus_i'].values[l]) - 1 
    j = int(case['branch']['bus_j'].values[l]) - 1
    
    i_B = i + nbus
    j_B = j + nbus

    M = np.zeros((D, D))
    
    # Real part block
    M[i, i] += y_sq[l]
    M[j, j] += y_sq[l]
    M[i, j] -= y_sq[l]
    M[j, i] -= y_sq[l]

    # Imaginary part block
    M[i_B, i_B] += y_sq[l]
    M[j_B, j_B] += y_sq[l]
    M[i_B, j_B] -= y_sq[l]
    M[j_B, i_B] -= y_sq[l]
    
    M_I.append(M)

# 2. Tensor Stack Generation
M_I_stack = torch.stack([
    torch.as_tensor(M_I[l], dtype=dtype, device=device) 
    for l in range(nbranch)
])


# Data for QCQP

## 1. Branch Flow Matrices (Equations 14-22)

In [28]:
# Initialize lists to store matrices for all branches
M_pf = []; M_qf = []; M_pt = []; M_qt = []

# Pre-calculate derived branch elements
g11 = g / (tau**2)
g12 = g * np.cos(theta_shift) / tau
g21 = g * np.sin(theta_shift) / tau
g22 = g

b11 = (b + bc/2) / (tau**2)
b12 = b * np.cos(theta_shift) / tau
b21 = b * np.sin(theta_shift) / tau
b22 = b + bc/2

for l in range(nbranch):
    # Python uses 0-based indexing; your dictionary offset to 1-based, so we subtract 1
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1
    
    # Real and Imaginary indices
    i_B = i + nbus
    j_B = j + nbus

    # --- FROM END ---
    A_pf = np.zeros((D, D))
    A_pf[i, i] = g11[l]
    A_pf[i_B, i_B] = g11[l]
    A_pf[i, j] = -(g12[l] - b21[l])
    A_pf[i_B, j_B] = -(g12[l] - b21[l])
    A_pf[i, j_B] = (g21[l] + b12[l])
    A_pf[i_B, j] = -(g21[l] + b12[l])
    M_pf.append(0.5 * (A_pf + A_pf.T))

    A_qf = np.zeros((D, D))
    A_qf[i, i] = -b11[l]
    A_qf[i_B, i_B] = -b11[l]
    A_qf[i, j] = (b12[l] + g21[l])
    A_qf[i_B, j_B] = (b12[l] + g21[l])
    A_qf[i, j_B] = -(b21[l] - g12[l])
    A_qf[i_B, j] = (b21[l] - g12[l])
    M_qf.append(0.5 * (A_qf + A_qf.T))

    # --- TO END ---
    A_pt = np.zeros((D, D))
    A_pt[j, j] = g22[l]
    A_pt[j_B, j_B] = g22[l]
    A_pt[j, i] = -(g12[l] + b21[l])
    A_pt[j_B, i_B] = -(g12[l] + b21[l])
    A_pt[j, i_B] = -(g21[l] - b12[l])
    A_pt[j_B, i] = (g21[l] - b12[l])
    M_pt.append(0.5 * (A_pt + A_pt.T))

    A_qt = np.zeros((D, D))
    A_qt[j, j] = -b22[l]
    A_qt[j_B, j_B] = -b22[l]
    A_qt[j, i] = (b12[l] - g21[l])
    A_qt[j_B, i_B] = (b12[l] - g21[l])
    A_qt[j, i_B] = (b21[l] + g12[l])
    A_qt[j_B, i] = -(b21[l] + g12[l])
    M_qt.append(0.5 * (A_qt + A_qt.T))

## 2. Nodal Power Injection Matrices (Equations 23-28)

In [29]:
# 1. Build Standard Y_bus matrices (G_bus and B_bus)
G_bus = np.zeros((nbus, nbus))
B_bus = np.zeros((nbus, nbus))

# Add shunts to the diagonals
np.fill_diagonal(G_bus, Gs)
np.fill_diagonal(B_bus, Bs)

# Add branch contributions
for l in range(nbranch):
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1

    # Diagonal elements
    G_bus[i, i] += g11[l]
    G_bus[j, j] += g22[l]
    B_bus[i, i] += b11[l]
    B_bus[j, j] += b22[l]

    # Off-diagonal elements (symmetric for the line itself, but tap ratios make Y_bus asymmetric)
    G_bus[i, j] += -g12[l]
    G_bus[j, i] += -g12[l] # Assuming simplified model where g12 roughly handles the tap phase
    B_bus[i, j] += -b12[l]
    B_bus[j, i] += -b12[l]

# 2. Build the QCQP Nodal Matrices
M_p = []; M_q = []

for i in range(nbus):
    A_pi = np.zeros((D, D))
    A_qi = np.zeros((D, D))
    
    # Active Power Row mappings (Eq 25)
    A_pi[i, :nbus] = G_bus[i, :]
    A_pi[i, nbus:] = -B_bus[i, :]
    A_pi[i + nbus, :nbus] = B_bus[i, :]
    A_pi[i + nbus, nbus:] = G_bus[i, :]
    M_p.append(0.5 * (A_pi + A_pi.T))
    
    # Reactive Power Row mappings (Eq 27)
    A_qi[i, :nbus] = -B_bus[i, :]
    A_qi[i, nbus:] = -G_bus[i, :]
    A_qi[i + nbus, :nbus] = G_bus[i, :]
    A_qi[i + nbus, nbus:] = -B_bus[i, :]
    M_q.append(0.5 * (A_qi + A_qi.T))

## 3. Branch Angle Difference & Voltage Matrices (Equations 29-34)

In [ ]:
M_c = []; M_s = []

for l in range(nbranch):
    i = int(case['branch']['bus_i'].values[l]) - 1
    j = int(case['branch']['bus_j'].values[l]) - 1
    i_B = i + nbus
    j_B = j + nbus

    # Angle Cosine Extraction (Eq 29 & 30)
    A_c = np.zeros((D, D))
    A_c[i, j] = 1
    A_c[i_B, j_B] = 1
    M_c.append(0.5 * (A_c + A_c.T))

    # Angle Sine Extraction (Eq 31 & 32)
    A_s = np.zeros((D, D))
    A_s[i_B, j] = 1
    A_s[i, j_B] = -1
    M_s.append(0.5 * (A_s + A_s.T))

M_V = []
for i in range(nbus):
    # Voltage Magnitude Extraction (Eq 33 & 34)
    A_V = np.zeros((D, D))
    A_V[i, i] = 1
    A_V[i + nbus, i + nbus] = 1
    M_V.append(A_V) # Already symmetric

## 4. Reference Angle Vector

In [31]:
# Identify the slack bus (MATPOWER sets bus type to 3 for slack)
slack_bus_idx = case['bus'][case['bus']['type'] == 3].index[0]

a_ref = np.zeros(D)
# Force the imaginary voltage component of the slack bus to 0
a_ref[slack_bus_idx + nbus] = 1

# Torch stack

## Tensor prep

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32

# ------------------------------------------------------------
# 1) Dimensions
# ------------------------------------------------------------
# These match the sizes from your MATPOWER data
nbus = nbus
ngen = ngen
nbranch = nbranch
d = D

# ------------------------------------------------------------
# 2) Stack quadratic matrices (For ALL buses and branches)
# ------------------------------------------------------------
# Nodal power and voltage matrices [nbus, d, d]
M_p_stack = torch.stack([torch.as_tensor(M_p[i], dtype=dtype, device=device) for i in range(nbus)])
M_q_stack = torch.stack([torch.as_tensor(M_q[i], dtype=dtype, device=device) for i in range(nbus)])
M_v_stack = torch.stack([torch.as_tensor(M_V[i], dtype=dtype, device=device) for i in range(nbus)])

# Branch flow matrices [nbranch, d, d]
M_pf_stack = torch.stack([torch.as_tensor(M_pf[l], dtype=dtype, device=device) for l in range(nbranch)])
M_qf_stack = torch.stack([torch.as_tensor(M_qf[l], dtype=dtype, device=device) for l in range(nbranch)])
M_pt_stack = torch.stack([torch.as_tensor(M_pt[l], dtype=dtype, device=device) for l in range(nbranch)])
M_qt_stack = torch.stack([torch.as_tensor(M_qt[l], dtype=dtype, device=device) for l in range(nbranch)])

# Angle difference matrices [nbranch, d, d]
M_c_stack = torch.stack([torch.as_tensor(M_c[l], dtype=dtype, device=device) for l in range(nbranch)])
M_s_stack = torch.stack([torch.as_tensor(M_s[l], dtype=dtype, device=device) for l in range(nbranch)])

# ------------------------------------------------------------
# 3) Symmetrize matrices (Required for stable Autograd)
# ------------------------------------------------------------
M_p_stack = 0.5 * (M_p_stack + M_p_stack.transpose(-1, -2))
M_q_stack = 0.5 * (M_q_stack + M_q_stack.transpose(-1, -2))
M_v_stack = 0.5 * (M_v_stack + M_v_stack.transpose(-1, -2))

M_pf_stack = 0.5 * (M_pf_stack + M_pf_stack.transpose(-1, -2))
M_qf_stack = 0.5 * (M_qf_stack + M_qf_stack.transpose(-1, -2))
M_pt_stack = 0.5 * (M_pt_stack + M_pt_stack.transpose(-1, -2))
M_qt_stack = 0.5 * (M_qt_stack + M_qt_stack.transpose(-1, -2))
M_c_stack = 0.5 * (M_c_stack + M_c_stack.transpose(-1, -2))
M_s_stack = 0.5 * (M_s_stack + M_s_stack.transpose(-1, -2))

# ------------------------------------------------------------
# 4) The C_g Matrix (Mapping Generators to Buses)
# ------------------------------------------------------------
# Shape: [nbus, ngen]
C_g = torch.zeros((nbus, ngen), dtype=dtype, device=device)
for gen_idx, bus_i in enumerate(case['gen']['bus_i'].values):
    bus_idx = int(bus_i) - 1 # convert to 0-based index
    C_g[bus_idx, gen_idx] = 1.0

# ------------------------------------------------------------
# 5) Vectors: Demands, Limits, and Reference
# ------------------------------------------------------------
Pd_bus = np.asarray(case['bus'].Pd.values, dtype=np.float32) / baseMVA
Qd_bus = np.asarray(case['bus'].Qd.values, dtype=np.float32) / baseMVA

pmax = np.asarray(case['gen'].Pmax.values, dtype=np.float32) / baseMVA
pmin = np.asarray(case['gen'].Pmin.values, dtype=np.float32) / baseMVA
qmax = np.asarray(case['gen'].Qmax.values, dtype=np.float32) / baseMVA
qmin = np.asarray(case['gen'].Qmin.values, dtype=np.float32) / baseMVA

# Apparent power branch limits (s_max)
smax = np.asarray(case['branch'].rateA.values, dtype=np.float32) / baseMVA
smax[smax == 0] = 9999.0  # Handle unconstrained lines gracefully

# Branch angle limits (converted to radians)
angmax = np.radians(np.asarray(case['branch'].angmax.values, dtype=np.float32))
angmin = np.radians(np.asarray(case['branch'].angmin.values, dtype=np.float32))

Vmax_arr = np.asarray(case['bus'].Vmax.values, dtype=np.float32)
Vmin_arr = np.asarray(case['bus'].Vmin.values, dtype=np.float32)

# ------------------------------------------------------------
# 6) Final problem dictionary for the PINN loss
# ------------------------------------------------------------
problem = {
    # Quadratic Matrices
    "M_p": M_p_stack,
    "M_q": M_q_stack,
    "M_v": M_v_stack,
    "M_pf": M_pf_stack,
    "M_qf": M_qf_stack,
    "M_pt": M_pt_stack,
    "M_qt": M_qt_stack,
    "M_c": M_c_stack,
    "M_s": M_s_stack,

    # Incidence Matrix
    "C_g": C_g,

    # Base Vectors
    "Pd": torch.as_tensor(Pd_bus, dtype=dtype, device=device),
    "Qd": torch.as_tensor(Qd_bus, dtype=dtype, device=device),
    
    "pmax": torch.as_tensor(pmax, dtype=dtype, device=device),
    "pmin": torch.as_tensor(pmin, dtype=dtype, device=device),
    "qmax": torch.as_tensor(qmax, dtype=dtype, device=device),
    "qmin": torch.as_tensor(qmin, dtype=dtype, device=device),
    
    "smax": torch.as_tensor(smax, dtype=dtype, device=device),
    "angmax": torch.as_tensor(angmax, dtype=dtype, device=device),
    "angmin": torch.as_tensor(angmin, dtype=dtype, device=device),
    
    "Vmax": torch.as_tensor(Vmax_arr, dtype=dtype, device=device),
    "Vmin": torch.as_tensor(Vmin_arr, dtype=dtype, device=device),
    
    # Anchor vector (Ensure a_ref from our earlier discussion is defined)
    "a_ref": torch.as_tensor(a_ref, dtype=dtype, device=device),

    # Metadata
    "nbus": nbus,
    "ngen": ngen,
    "nbranch": nbranch
}

print("Constructed PINN problem data for QCQP:")
print(f"  nbus    = {nbus}")
print(f"  ngen    = {ngen}")
print(f"  nbranch = {nbranch}")
print(f"  M_pf, M_qf shape = {tuple(problem['M_pf'].shape)}, {tuple(problem['M_qf'].shape)}")
print(f"  M_pt, M_qt shape = {tuple(problem['M_pt'].shape)}, {tuple(problem['M_qt'].shape)}")
print(f"  C_g shape  = {tuple(problem['C_g'].shape)}")
print(f"  M_p, M_q shape  = {tuple(problem['M_p'].shape)}, {tuple(problem['M_q'].shape)}")
print(f"  M_s, M_c shape  = {tuple(problem['M_s'].shape)}, {tuple(problem['M_c'].shape)}")
print(f"  M_V shape  = {tuple(problem['M_v'].shape)}")

Constructed PINN problem data for QCQP:
  nbus    = 3
  ngen    = 3
  nbranch = 3
  M_pf, M_qf shape = (3, 6, 6), (3, 6, 6)
  M_pt, M_qt shape = (3, 6, 6), (3, 6, 6)
  C_g shape  = (3, 3)
  M_p, M_q shape  = (3, 6, 6), (3, 6, 6)
  M_s, M_c shape  = (3, 6, 6), (3, 6, 6)
  M_V shape  = (3, 6, 6)


# Random demand

In [40]:
def gaussian_batch(base_tensor, batch_size, variation_std=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with Gaussian random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation_std (float): standard deviation of Gaussian noise, relative to base_tensor
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    # Expand base tensor to batch
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Gaussian noise scaled by base_tensor
    noise = variation_std * base_tensor.unsqueeze(0) * torch.randn_like(base_batch)
    
    # Perturbed batch
    batch = base_batch + noise
    
    # Clamp if necessary
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch

def uniform_batch(base_tensor, batch_size, variation=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with uniform random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation (float): maximum relative deviation (±variation)
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Uniform noise in [-variation, +variation]
    noise = (2 * torch.rand_like(base_batch) - 1) * variation
    
    batch = base_batch * (1 + noise)
    
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch


In [56]:
import torch

def gaussian_batch(base_tensor, batch_size, variation_std=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with Gaussian random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation_std (float): standard deviation of Gaussian noise, relative to base_tensor
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    # Expand base tensor to batch
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Gaussian noise scaled by base_tensor
    noise = variation_std * base_tensor.unsqueeze(0) * torch.randn_like(base_batch)
    
    # Perturbed batch
    batch = base_batch + noise
    
    # Clamp if necessary
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch


def uniform_batch(base_tensor, batch_size, variation=0.05, clamp_min=0.0):
    """
    Create a batch of tensors with uniform random variations.
    
    Args:
        base_tensor (torch.Tensor): original tensor of shape [N]
        batch_size (int): number of samples in batch
        variation (float): maximum relative deviation (±variation)
        clamp_min (float): minimum allowed value for perturbed tensor
    
    Returns:
        torch.Tensor: batch of shape [batch_size, N]
    """
    base_batch = base_tensor.unsqueeze(0).repeat(batch_size, 1)
    
    # Uniform noise in [-variation, +variation]
    noise = (2 * torch.rand_like(base_batch) - 1) * variation
    
    batch = base_batch * (1 + noise)
    
    if clamp_min is not None:
        batch = torch.clamp(batch, min=clamp_min)
    
    return batch

# Base line model

## 1. PINN Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

class FeasibleQCQPMLP(nn.Module):
    """
    Input:
        Pd: [B, nbus]
        Qd: [B, nbus]
    Output:
        v:  [B, 2*nbus] (Rectangular voltages)
        pg: [B, ngen]   (Active generation)
        qg: [B, ngen]   (Reactive generation)
    """
    def __init__(self, nbus: int, ngen: int, slack_imag_idx: int, hidden: int = 512):
        super().__init__()
        self.nbus = nbus
        self.ngen = ngen
        self.in_dim = 2 * nbus
        self.out_dim_v = 2 * nbus
        self.out_dim_g = 2 * ngen 
        self.slack_imag_idx = int(slack_imag_idx)

        # Core MLP
        self.net = nn.Sequential(
            nn.Linear(self.in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, self.out_dim_v + self.out_dim_g),
        )

    def forward(self, Pd: torch.Tensor, Qd: torch.Tensor, problem: dict) -> tuple:
        B = Pd.shape[0]
        x = torch.cat([Pd, Qd], dim=-1)
        raw = self.net(x)

        # 1. Slice outputs
        v_raw = raw[:, :self.out_dim_v]
        g_raw = raw[:, self.out_dim_v:]
        
        pg_raw = g_raw[:, :self.ngen]
        qg_raw = g_raw[:, self.ngen:]

        # 2. Bound Voltages to [-Vmax, Vmax] using Tanh for smooth gradients
        Vmax_b = problem["Vmax"].reshape(1, -1).expand(B, -1)
        Vmax_full = torch.cat([Vmax_b, Vmax_b], dim=-1) # For real and imaginary parts
        v = torch.tanh(v_raw) * Vmax_full

        # Constraint (2m): Enforce slack imaginary part = 0 exactly
        v_clone = v.clone()
        v_clone[:, self.slack_imag_idx] = 0.0
        v = v_clone

        # 3. Bound Generation strictly between [min, max] using Sigmoid
        pmax_b = problem["pmax"].reshape(1, -1).expand(B, -1)
        pmin_b = problem["pmin"].reshape(1, -1).expand(B, -1)
        qmax_b = problem["qmax"].reshape(1, -1).expand(B, -1)
        qmin_b = problem["qmin"].reshape(1, -1).expand(B, -1)

        pg = pmin_b + torch.sigmoid(pg_raw) * (pmax_b - pmin_b)
        qg = qmin_b + torch.sigmoid(qg_raw) * (qmax_b - qmin_b)

        return v, pg, qg

## 2. The QCQP Loss Function

In [42]:
def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [K, d, d] -> [B, K]
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def compute_qcqp_loss(model: nn.Module, Pd_batch: torch.Tensor, Qd_batch: torch.Tensor, problem: dict, weights: dict):
    B = Pd_batch.shape[0]
    
    # Predict continuous variables (bounded by construction)
    v, pg, qg = model(Pd_batch, Qd_batch, problem)

    # --------------------------------------------------------
    # Unpack Problem Matrices
    # --------------------------------------------------------
    M_p, M_q = problem["M_p"], problem["M_q"]
    M_pf, M_qf = problem["M_pf"], problem["M_qf"]
    M_pt, M_qt = problem["M_pt"], problem["M_qt"]
    M_c, M_s, M_v = problem["M_c"], problem["M_s"], problem["M_v"]
    C_g = problem["C_g"] # [nbus, ngen]
    
    smax = problem["smax"].unsqueeze(0).expand(B, -1)
    angmax = problem["angmax"].unsqueeze(0).expand(B, -1)
    angmin = problem["angmin"].unsqueeze(0).expand(B, -1)
    Vmin = problem["Vmin"].unsqueeze(0).expand(B, -1)
    Vmax = problem["Vmax"].unsqueeze(0).expand(B, -1)

    # --------------------------------------------------------
    # Evaluate Quadratic Forms
    # --------------------------------------------------------
    vp = quad_batch_stack(v, M_p)   # [B, nbus]
    vq = quad_batch_stack(v, M_q)
    
    pf = quad_batch_stack(v, M_pf)  # [B, nbranch]
    qf = quad_batch_stack(v, M_qf)
    pt = quad_batch_stack(v, M_pt)
    qt = quad_batch_stack(v, M_qt)
    
    vc = quad_batch_stack(v, M_c)
    vs = quad_batch_stack(v, M_s)
    vv = quad_batch_stack(v, M_v)

    # --------------------------------------------------------
    # Constraints Mapping (Equations 2h - 2m)
    # --------------------------------------------------------
    
    # 1. Nodal Power Balance (Eq 2h & 2i): [C_g * pg]_i - pd_i = v^T M_p v
    # pg @ C_g.T automatically maps the ngen generations to their nbus locations
    h_p = (pg @ C_g.T) - Pd_batch - vp
    h_q = (qg @ C_g.T) - Qd_batch - vq

    # 2. Branch Thermal Limits (Eq 2f & 2g): p^2 + q^2 <= smax^2
    g_sf = (pf**2 + qf**2) - smax**2
    g_st = (pt**2 + qt**2) - smax**2

    # 3. Angle Difference Stability (Eq 2j & 2k)
    g_ang_min = torch.tan(angmin) * vc - vs
    g_ang_max = vs - torch.tan(angmax) * vc

    # 4. Voltage Magnitude Security (Eq 2l): Vmin^2 <= v^T M_v v <= Vmax^2
    g_v_max = vv - (Vmax**2)
    g_v_min = (Vmin**2) - vv

    # 5. Objective (Eq 2a) - Minimize total active generation
    # Simplified here to sum(p_g); can be replaced with c1^T p_g + c2^T(pg*pg) if cost coefficients exist
    obj = pg.sum(dim=1).mean()

    # --------------------------------------------------------
    # Compute Penalties
    # --------------------------------------------------------
    loss_eq_p = h_p.pow(2).mean()
    loss_eq_q = h_q.pow(2).mean()
    
    loss_thermal = F.relu(g_sf).pow(2).mean() + F.relu(g_st).pow(2).mean()
    loss_ang = F.relu(g_ang_min).pow(2).mean() + F.relu(g_ang_max).pow(2).mean()
    loss_v = F.relu(g_v_max).pow(2).mean() + F.relu(g_v_min).pow(2).mean()

    total_loss = (
        weights["eq_p"] * loss_eq_p +
        weights["eq_q"] * loss_eq_q +
        weights["thermal"] * loss_thermal +
        weights["ang"] * loss_ang +
        weights["v"] * loss_v +
        weights["obj"] * obj
    )

    diagnostics = {
        "loss_total": total_loss.detach().item(),
        "loss_eq_p": loss_eq_p.detach().item(),
        "loss_eq_q": loss_eq_q.detach().item(),
        "loss_thermal": loss_thermal.detach().item(),
        "max_h_p_viol": h_p.abs().max().detach().item(),
        "max_thermal_viol": torch.max(F.relu(g_sf).max(), F.relu(g_st).max()).detach().item(),
        "obj_val": obj.detach().item()
    }

    return total_loss, diagnostics

## 3. Training

In [43]:
# Extract the exact index where a_ref is 1 (the slack bus imaginary component)
slack_imag_idx = (problem["a_ref"] == 1).nonzero(as_tuple=True)[0].item()

model = FeasibleQCQPMLP(
    nbus=problem["nbus"],
    ngen=problem["ngen"],
    slack_imag_idx=slack_imag_idx
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Define penalty weights (tune these based on convergence)
loss_weights = {
    "eq_p": 10.0,
    "eq_q": 10.0,
    "thermal": 5.0,
    "ang": 1.0,
    "v": 5.0,
    "obj": 0.01
}

# Example Training Loop
for epoch in range(1000):
    # Base demands from problem dict
    Pd_base = problem["Pd"]
    Qd_base = problem["Qd"]
    
    # Generate random scenario batch
    Pd_batch = gaussian_batch(Pd_base, batch_size=32)
    Qd_batch = gaussian_batch(Qd_base, batch_size=32)
    
    optimizer.zero_grad()
    loss, diag = compute_qcqp_loss(model, Pd_batch, Qd_batch, problem, loss_weights)
    loss.backward()
    
    torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
    optimizer.step()
    
    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | Loss: {diag['loss_total']:.4f} | Max P-Balance Error: {diag['max_h_p_viol']:.4f}")

Epoch    0 | Loss: 557.2088 | Max P-Balance Error: 9.4321
Epoch  100 | Loss: 0.3069 | Max P-Balance Error: 0.4111
Epoch  200 | Loss: 0.1244 | Max P-Balance Error: 0.1529
Epoch  300 | Loss: 0.1842 | Max P-Balance Error: 0.2115
Epoch  400 | Loss: 0.3157 | Max P-Balance Error: 0.1879
Epoch  500 | Loss: 0.1145 | Max P-Balance Error: 0.1269
Epoch  600 | Loss: 0.1215 | Max P-Balance Error: 0.1839
Epoch  700 | Loss: 0.0434 | Max P-Balance Error: 0.0545
Epoch  800 | Loss: 0.1914 | Max P-Balance Error: 0.1187
Epoch  900 | Loss: 0.0702 | Max P-Balance Error: 0.0658


# Old

In [57]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def quad_batch(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [d, d] -> [B]
    return torch.einsum("bi,ij,bj->b", v, M, v)

def quad_batch_stack(v: torch.Tensor, M: torch.Tensor) -> torch.Tensor:
    # v: [B, d], M: [K, d, d] -> [B, K]
    return torch.einsum("bi,kij,bj->bk", v, M, v)

def gaussian_batch(x: torch.Tensor, batch_size: int, variation_std: float = 0.05) -> torch.Tensor:
    # x: [n] -> [B, n]
    x = x.reshape(1, -1).expand(batch_size, -1)
    scale = torch.clamp(x.abs(), min=1.0)
    noise = variation_std * torch.randn_like(x) * scale
    return x + noise

def to_row_batch(x: torch.Tensor, B: int) -> torch.Tensor:
    # robust expansion for vectors that may be [N], [N,1], [1,N], etc.
    return x.reshape(1, -1).expand(B, -1)


# ------------------------------------------------------------
# Verification-oriented feasibility network
# ------------------------------------------------------------

class FeasibleVoltageMLP(nn.Module):
    """
    Input:
        Pd: [B, nbus]
        Qd: [B, nbus]

    Output:
        v:  [B, 2*nbus]

    Architecture choices for verification-readiness:
      - only affine + ReLU + clamp
      - no dual variables
      - no implicit optimization layer
      - no eig/SVD/nullspace operations
      - slack imaginary part fixed to zero by construction
    """
    def __init__(
        self,
        nbus: int,
        hidden1: int = 256,
        hidden2: int = 256,
        hidden3: int = 256,
        slack_imag_idx: int = None,
        bound_margin: float = 0.98,   # keep outputs slightly inside box
    ):
        super().__init__()
        self.nbus = nbus
        self.in_dim = 2 * nbus
        self.out_dim = 2 * nbus
        self.slack_imag_idx = nbus if slack_imag_idx is None else int(slack_imag_idx)
        self.bound_margin = float(bound_margin)

        self.net = nn.Sequential(
            nn.Linear(self.in_dim, hidden1),
            nn.ReLU(),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Linear(hidden2, hidden3),
            nn.ReLU(),
            nn.Linear(hidden3, self.out_dim),
        )

    def forward(self, Pd: torch.Tensor, Qd: torch.Tensor, Vmax: torch.Tensor) -> torch.Tensor:
        """
        Vmax: [nbus] or [B, nbus]
        Returns v = [vr, vi] with each component bounded in [-bound_margin*Vmax, +bound_margin*Vmax]
        and slack imaginary component exactly zero.
        """
        B = Pd.shape[0]
        x = torch.cat([Pd, Qd], dim=-1)            # [B, 2*nbus]
        raw = self.net(x)                          # [B, 2*nbus]

        vr_raw = raw[:, :self.nbus]                # [B, nbus]
        vi_raw = raw[:, self.nbus:]                # [B, nbus]

        Vmax_b = Vmax.reshape(1, -1).expand(B, -1) if Vmax.dim() == 1 else Vmax
        comp_bound = self.bound_margin * Vmax_b

        # Piecewise-linear bounded map: clamp is simpler than tanh/sigmoid for verification-oriented use
        vr = torch.clamp(vr_raw, min=-comp_bound, max=comp_bound)
        vi = torch.clamp(vi_raw, min=-comp_bound, max=comp_bound)

        # Enforce slack imaginary part = 0 exactly
        vi = vi.clone()
        vi[:, self.slack_imag_idx - self.nbus if self.slack_imag_idx >= self.nbus else 0] = 0.0
        # In your previous Ms, Ms[nbus, nbus] = 1, so slack_imag_idx = nbus means first imag entry.
        # The expression above converts the global index into the local imag-block index.

        v = torch.cat([vr, vi], dim=-1)            # [B, 2*nbus]
        return v


# ------------------------------------------------------------
# Pure feasibility loss
# ------------------------------------------------------------

def compute_feasibility_loss(
    model: nn.Module,
    Pd_batch: torch.Tensor,
    Qd_batch: torch.Tensor,
    problem: dict,
    w_eq_p: float = 10.0,
    w_eq_q: float = 10.0,
    w_ineq_gen: float = 5.0,
    w_ineq_v: float = 5.0,
    w_ineq_I: float = 5.0,
    w_slack: float = 20.0,
    w_obj: float = 0.01,
):
    B = Pd_batch.shape[0]
    device = Pd_batch.device

    # --------------------------------------------------------
    # pull problem data
    # --------------------------------------------------------
    M_c      = problem["M_c"]          # [d,d]
    M_p_load = problem["M_p_load"]     # [nload,d,d]
    M_q_load = problem["M_q_load"]     # [nload,d,d]
    M_p_gen  = problem["M_p_gen"]      # [ngen,d,d]
    M_q_gen  = problem["M_q_gen"]      # [ngen,d,d]
    M_v      = problem["M_v"]          # [nbus,d,d]
    M_I      = problem["M_I"]          # [nbranch,d,d]
    M_s      = problem["M_s"]          # [d,d]

    load_bus_pos = problem["load_bus_pos"]   # [nload]
    gen_bus_pos  = problem["gen_bus_pos"]    # [ngen]

    pmax_g = to_row_batch(problem["pmax_g"], B)   # [B,ngen]
    pmin_g = to_row_batch(problem["pmin_g"], B)
    qmax_g = to_row_batch(problem["qmax_g"], B)
    qmin_g = to_row_batch(problem["qmin_g"], B)
    Vmax   = to_row_batch(problem["Vmax"], B)     # [B,nbus]
    Vmin   = to_row_batch(problem["Vmin"], B)
    Imax   = to_row_batch(problem["Imax"], B)     # [B,nbranch]

    # --------------------------------------------------------
    # sample-dependent demand values
    # --------------------------------------------------------
    Pd_load = Pd_batch[:, load_bus_pos]      # [B,nload]
    Qd_load = Qd_batch[:, load_bus_pos]      # [B,nload]
    Pd_gen  = Pd_batch[:, gen_bus_pos]       # [B,ngen]
    Qd_gen  = Qd_batch[:, gen_bus_pos]       # [B,ngen]

    # --------------------------------------------------------
    # predict primal voltage
    # --------------------------------------------------------
    v = model(Pd_batch, Qd_batch, Vmax=problem["Vmax"].reshape(-1))

    # --------------------------------------------------------
    # constraint values
    # --------------------------------------------------------
    vp_load = quad_batch_stack(v, M_p_load)   # [B,nload]
    vq_load = quad_batch_stack(v, M_q_load)   # [B,nload]
    vp_gen  = quad_batch_stack(v, M_p_gen)    # [B,ngen]
    vq_gen  = quad_batch_stack(v, M_q_gen)    # [B,ngen]
    vv      = quad_batch_stack(v, M_v)        # [B,nbus]
    vI      = quad_batch_stack(v, M_I)        # [B,nbranch]
    vs      = quad_batch(v, M_s).unsqueeze(-1)

    # Equalities on non-generator buses
    h_pb = vp_load + Pd_load
    h_qb = vq_load + Qd_load

    # Generator inequalities
    g_pg_u = vp_gen - (pmax_g - Pd_gen)
    g_pg_l = -vp_gen + (pmin_g - Pd_gen)
    g_qg_u = vq_gen - (qmax_g - Qd_gen)
    g_qg_l = -vq_gen + (qmin_g - Qd_gen)

    # Voltage / current inequalities
    g_v_u = vv - Vmax.pow(2)
    g_v_l = -vv + Vmin.pow(2)
    g_I   = vI - Imax.pow(2)

    # Slack equality
    h_s = vs

    # --------------------------------------------------------
    # residual losses
    # --------------------------------------------------------
    loss_eq_p = h_pb.pow(2).mean()
    loss_eq_q = h_qb.pow(2).mean()

    loss_gen_ineq = (
        F.relu(g_pg_u).pow(2).mean()
        + F.relu(g_pg_l).pow(2).mean()
        + F.relu(g_qg_u).pow(2).mean()
        + F.relu(g_qg_l).pow(2).mean()
    )

    loss_v_ineq = (
        F.relu(g_v_u).pow(2).mean()
        + F.relu(g_v_l).pow(2).mean()
    )

    loss_I_ineq = F.relu(g_I).pow(2).mean()

    # h_s should already be zero by construction if Ms selects the slack imaginary component,
    # but keep this term as a safety check.
    loss_slack = h_s.pow(2).mean()

    # small objective regularizer (optional, to bias toward cheaper feasible points)
    obj = quad_batch(v, M_c).mean()

    total_loss = (
        w_eq_p * loss_eq_p
        + w_eq_q * loss_eq_q
        + w_ineq_gen * loss_gen_ineq
        + w_ineq_v * loss_v_ineq
        + w_ineq_I * loss_I_ineq
        + w_slack * loss_slack
        + w_obj * obj
    )

    diagnostics = {
        "loss_total": total_loss.detach(),
        "loss_eq_p": loss_eq_p.detach(),
        "loss_eq_q": loss_eq_q.detach(),
        "loss_gen_ineq": loss_gen_ineq.detach(),
        "loss_v_ineq": loss_v_ineq.detach(),
        "loss_I_ineq": loss_I_ineq.detach(),
        "loss_slack": loss_slack.detach(),
        "obj": obj.detach(),
        "max_abs_h_pb": h_pb.abs().max().detach(),
        "max_abs_h_qb": h_qb.abs().max().detach(),
        "max_g_pg_u": g_pg_u.max().detach(),
        "max_g_pg_l": g_pg_l.max().detach(),
        "max_g_qg_u": g_qg_u.max().detach(),
        "max_g_qg_l": g_qg_l.max().detach(),
        "max_g_v_u": g_v_u.max().detach(),
        "max_g_v_l": g_v_l.max().detach(),
        "max_g_I": g_I.max().detach(),
    }

    return total_loss, diagnostics


# ------------------------------------------------------------
# Build model
# ------------------------------------------------------------

device = problem["M_c"].device
nbus = len(problem["all_bus_ids"])

# In your previous construction Ms[nbus, nbus] = 1, so global slack imag index is nbus
slack_imag_idx = nbus

model = FeasibleVoltageMLP(
    nbus=nbus,
    hidden1=10*nbus,
    hidden2=5*nbus,
    hidden3=10*nbus,
    slack_imag_idx=slack_imag_idx,
    bound_margin=0.98,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)

Pd_bus_base = problem["Pd_bus"].reshape(-1)
Qd_bus_base = problem["Qd_bus"].reshape(-1)




In [58]:
# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------

num_epochs = 3000
batch_size = 32
best_loss = float("inf")
best_state = None

for epoch in range(num_epochs):
    # sample demand scenarios
    Pd_batch = gaussian_batch(Pd_bus_base, batch_size=batch_size, variation_std=0.05)
    Qd_batch = gaussian_batch(Qd_bus_base, batch_size=batch_size, variation_std=0.05)

    # demands should remain nonnegative
    Pd_batch = torch.clamp(Pd_batch, min=0.0)
    Qd_batch = torch.clamp(Qd_batch, min=0.0)

    optimizer.zero_grad()

    loss, diagnostics = compute_feasibility_loss(
        model=model,
        Pd_batch=Pd_batch,
        Qd_batch=Qd_batch,
        problem=problem,
        w_eq_p=10.0,
        w_eq_q=10.0,
        w_ineq_gen=5.0,
        w_ineq_v=5.0,
        w_ineq_I=5.0,
        w_slack=20.0,
        w_obj=0.01,
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
    optimizer.step()
    scheduler.step()

    loss_value = loss.detach().item()
    if loss_value < best_loss:
        best_loss = loss_value
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 100 == 0:
        print(f"\nEpoch {epoch:5d} | loss = {loss_value:.6e}")
        print({k: v.detach().item() for k, v in diagnostics.items()})

# restore best model
if best_state is not None:
    model.load_state_dict(best_state)

print("\nBest training loss:", best_loss)





Epoch     0 | loss = 2.054241e+02
{'loss_total': 205.42413330078125, 'loss_eq_p': 18.244476318359375, 'loss_eq_q': 2.292607545852661, 'loss_gen_ineq': 0.006596775725483894, 'loss_v_ineq': 1.1064282823269878e-08, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': 2.030914545059204, 'max_abs_h_pb': 22.389909744262695, 'max_abs_h_qb': 5.310649871826172, 'max_g_pg_u': -3.711991786956787, 'max_g_pg_l': 0.1108459010720253, 'max_g_qg_u': -3.16375470161438, 'max_g_qg_l': -2.246126413345337, 'max_g_v_u': 0.00011141680442960933, 'max_g_v_l': -3.52084098267369e-05, 'max_g_I': -1291322.375}

Epoch   100 | loss = 2.060926e+02
{'loss_total': 206.09263610839844, 'loss_eq_p': 18.309724807739258, 'loss_eq_q': 2.297386884689331, 'loss_gen_ineq': 0.00825719628483057, 'loss_v_ineq': 1.2206811561554787e-08, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': -1.9756052494049072, 'max_abs_h_pb': 22.377777099609375, 'max_abs_h_qb': 5.425243377685547, 'max_g_pg_u': -3.663360834121704, 'max_g_pg_l': 0.1697320193052292, 

In [43]:
# ------------------------------------------------------------
# Final evaluation on one fresh batch
# ------------------------------------------------------------

with torch.no_grad():
    Pd_test = gaussian_batch(Pd_bus_base, batch_size=64, variation_std=0.05)
    Qd_test = gaussian_batch(Qd_bus_base, batch_size=64, variation_std=0.05)
    Pd_test = torch.clamp(Pd_test, min=0.0)
    Qd_test = torch.clamp(Qd_test, min=0.0)

    test_loss, test_diag = compute_feasibility_loss(
        model=model,
        Pd_batch=Pd_test,
        Qd_batch=Qd_test,
        problem=problem,
        w_eq_p=10.0,
        w_eq_q=10.0,
        w_ineq_gen=5.0,
        w_ineq_v=5.0,
        w_ineq_I=5.0,
        w_slack=20.0,
        w_obj=0.01,
    )

print("\nFinal test diagnostics:")
print({k: v.detach().item() for k, v in test_diag.items()})

# The trained object you want for later verification is `model`.
# A verification pipeline can then check, over an input demand set,
# whether the resulting v satisfies:
#   h_pb = 0, h_qb = 0,
#   g_pg_u <= 0, g_pg_l <= 0,
#   g_qg_u <= 0, g_qg_l <= 0,
#   g_v_u <= 0, g_v_l <= 0,
#   g_I <= 0,
#   h_s = 0.


Final test diagnostics:
{'loss_total': 204.63502502441406, 'loss_eq_p': 18.165790557861328, 'loss_eq_q': 2.294523239135742, 'loss_gen_ineq': 0.007044778671115637, 'loss_v_ineq': 1.2206812449733206e-08, 'loss_I_ineq': 0.0, 'loss_slack': 0.0, 'obj': -0.3332103192806244, 'max_abs_h_pb': 21.82909393310547, 'max_abs_h_qb': 5.523430824279785, 'max_g_pg_u': -3.662353277206421, 'max_g_pg_l': 0.11239556223154068, 'max_g_qg_u': -3.145278215408325, 'max_g_qg_l': -2.25, 'max_g_v_u': 0.00011141680442960933, 'max_g_v_l': -3.52084098267369e-05, 'max_g_I': -430440.75}
